# Test Data Preparation

This notebook merges the weather and PV power data for the test year (third year: July 2012 - June 2013) and saves the combined dataset for model evaluation.

## Merge Test Dataset

Load the weather (WX) and photovoltaic (PV) Excel files, extract the third year sheet, merge them, and save as CSV and Excel formats.

In [1]:
import sys
import os

# Add project root to path for imports
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import datetime

# File paths
WX_PATH = "../data/raw/wx_ds_final.xlsx"
PV_PATH = "../data/raw/pv_ds_final.xlsx"
OUTPUT_CSV = "../data/processed/merged_test_ds.csv"
OUTPUT_EXCEL = "../data/processed/merged_test_ds.xlsx"

# Load Excel files to get sheet names
xls_wx = pd.ExcelFile(WX_PATH)
xls_pv = pd.ExcelFile(PV_PATH)

print(f"WX sheets: {xls_wx.sheet_names}")
print(f"PV sheets: {xls_pv.sheet_names}")

# Load only the last sheet (third year - test data)
last_sheet_wx = xls_wx.sheet_names[-1]
last_sheet_pv = xls_pv.sheet_names[-1]

print(f"\nLoading WX sheet: '{last_sheet_wx}'")
print(f"Loading PV sheet: '{last_sheet_pv}'")

df_wx = pd.read_excel(xls_wx, sheet_name=last_sheet_wx)
df_pv = pd.read_excel(xls_pv, sheet_name=last_sheet_pv)

print(f"\nWX shape: {df_wx.shape}")
print(f"PV shape: {df_pv.shape}")
print(f"WX columns: {df_wx.columns.tolist()}")
print(f"PV columns: {df_pv.columns.tolist()}")

# Rename PV columns
df_pv.columns = ["datetime", "pv_power"]

# Merge datasets
df_merged = pd.concat([df_wx, df_pv], axis=1)
df_merged.drop(columns=["datetime"], inplace=True)  # Use dt_iso instead

# Fix timezone (Sydney +10)
df_merged["dt_iso"] = pd.to_datetime(df_merged["dt_iso"], utc=True, errors="raise")
tz_fixed = datetime.timezone(datetime.timedelta(hours=10))
df_merged["dt_iso"] = df_merged["dt_iso"].dt.tz_convert(tz_fixed)

print(f"\nMerged dataset shape: {df_merged.shape}")
print(f"Date range: {df_merged['dt_iso'].min()} -> {df_merged['dt_iso'].max()}")

# Save as CSV
df_merged.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}")

# Save as Excel (remove timezone for compatibility)
df_merged_excel = df_merged.copy()
df_merged_excel["dt_iso"] = df_merged_excel["dt_iso"].dt.tz_localize(None)
df_merged_excel.to_excel(OUTPUT_EXCEL, index=False)
print(f"Saved: {OUTPUT_EXCEL}")

WX sheets: ['07-10--06-11', '07-11--06-12', '07-12--06-13']
PV sheets: ['07-10--06-11', '07-11--06-12', '07-12--06-13']

Loading WX sheet: '07-12--06-13'
Loading PV sheet: '07-12--06-13'

WX shape: (8760, 15)
PV shape: (8760, 2)
WX columns: ['dt_iso', 'lat', 'lon', 'temp', 'dew_point', 'pressure', 'humidity', 'wind_speed', 'wind_deg', 'rain_1h', 'clouds_all', 'weather_description', 'Dhi', 'Dni', 'Ghi']
PV columns: ['Max kWp', 82.41]

Merged dataset shape: (8760, 16)
Date range: 2012-07-01 00:00:00+10:00 -> 2013-06-30 23:00:00+10:00

Saved: ../data/processed/merged_test_ds.csv
Saved: ../data/processed/merged_test_ds.xlsx


## Compare Weather Categories

Analyze the weather description categories in both training and test datasets to verify consistency and identify any new categories in the test set.

In [6]:
import pandas as pd

TRAIN_PATH = "../data/processed/merge_ds.csv"
TEST_PATH = "../data/processed/merged_test_ds.csv"

list_paths = [TRAIN_PATH, TEST_PATH]

for p in list_paths:
    if p.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(p)
    else:
        df = pd.read_csv(p)

    # Count weather description categories
    print(f"\nFile: {p}")
    print(f"Shape: {df.shape}")
    print(f"\n{'='*50}")
    print("CATEGORIES IN weather_description:")
    print('='*50)

    counts = df['weather_description'].value_counts()
    print(f"\nTotal unique categories: {len(counts)}\n")
    print(counts.to_string())


File: ../data/processed/merge_ds.csv
Shape: (17544, 16)

CATEGORIES IN weather_description:

Total unique categories: 19

weather_description
sky is clear                   4264
light rain                     3068
overcast clouds                3027
scattered clouds               2619
broken clouds                  2242
few clouds                     1462
moderate rain                   608
haze                            113

File: ../data/processed/merged_test_ds.csv
Shape: (8760, 16)

CATEGORIES IN weather_description:

Total unique categories: 20

weather_description
sky is clear                    2315
overcast clouds                 1322
scattered clouds                1226
light rain                      1023
broken clouds                    983
few clouds                       787
haze                             684
moderate rain                    294


## Inspect Test Data Categories

Detailed inspection of weather categories in the test dataset only.

In [4]:
import pandas as pd

# Load test file
FILE_PATH = "../data/processed/merged_test_ds.csv"

if FILE_PATH.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(FILE_PATH)
else:
    df = pd.read_csv(FILE_PATH)

# Count weather description categories
print(f"File: {FILE_PATH}")
print(f"Shape: {df.shape}")
print(f"\n{'='*50}")
print("CATEGORIES IN weather_description:")
print('='*50)

counts = df['weather_description'].value_counts()
print(f"\nTotal unique categories: {len(counts)}\n")
print(counts.to_string())

File: ../data/processed/merged_test_ds.csv
Shape: (8760, 16)

CATEGORIES IN weather_description:

Total unique categories: 20

weather_description
sky is clear                    2315
overcast clouds                 1322
scattered clouds                1226
light rain                      1023
broken clouds                    983
few clouds                       787
haze                             684
moderate rain                    294
